# RiverGuard: dense-trash classifier (Kaggle)

Whole-photo classifier (`dense` vs `not_dense`) trained on the dataset built by `training/build_density.py`,
attached as a Kaggle dataset. Everything in `/kaggle/working/output` is what gets downloaded.

In [ ]:
BASE_WEIGHTS = "yolo11s-cls.pt"   # ImageNet-pretrained
EPOCHS = 60
PATIENCE = 15
IMGSZ = 384
BATCH = 32
ULTRALYTICS_VERSION = "8.4.163"

In [ ]:
%pip install -q "ultralytics=={ULTRALYTICS_VERSION}"

In [ ]:
import shutil, zipfile
from pathlib import Path
DATA = Path("/kaggle/working/data")
if not (DATA / "train").exists():
    found = [p.parent for p in Path("/kaggle/input").rglob("meta.csv")]
    if found:
        shutil.copytree(found[0], DATA, dirs_exist_ok=True)
    else:
        zipfile.ZipFile(next(Path("/kaggle/input").rglob("*.zip"))).extractall(DATA)
print({s: {c: len(list((DATA / s / c).glob("*"))) for c in ("dense", "not_dense")} for s in ("train", "val", "test")})

In [ ]:
import torch
from ultralytics import YOLO
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (enable GPU in settings!)")
model = YOLO(BASE_WEIGHTS)
model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, device=0, workers=4,
            seed=0, cos_lr=True, project="/kaggle/working/runs", name="density", exist_ok=True)

In [ ]:
# Per-image dense probabilities on val and test, so the alert threshold can be chosen on val and reported on test.
import csv, json
best = YOLO("/kaggle/working/runs/density/weights/best.pt")
dense_idx = [i for i, n in best.names.items() if n == "dense"][0]
rows = []
for split in ("val", "test"):
    for cls in ("dense", "not_dense"):
        for f in sorted((DATA / split / cls).glob("*.jpg")):
            p = float(best(str(f), imgsz=IMGSZ, verbose=False)[0].probs.data[dense_idx])
            rows.append({"split": split, "file": f"{split}/{cls}/{f.name}", "true": cls, "p_dense": round(p, 4)})
out = Path("/kaggle/working/output"); out.mkdir(exist_ok=True)
with open(out / "scores.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
for split in ("val", "test"):
    r = [x for x in rows if x["split"] == split]
    acc = sum((x["p_dense"] >= 0.5) == (x["true"] == "dense") for x in r) / len(r)
    print(split, len(r), "images, accuracy at 0.5:", round(acc, 3))

In [ ]:
run = Path("/kaggle/working/runs/density")
shutil.copy(run / "weights" / "best.pt", out / "density.pt")
for f in ["results.csv", "results.png", "confusion_matrix.png", "args.yaml"]:
    if (run / f).exists():
        shutil.copy(run / f, out / f)
shutil.rmtree(DATA)
print(sorted(p.name for p in out.iterdir()))